In [0]:
spark.sql("USE workspace.ventes")

spark.sql("""
CREATE OR REPLACE TEMP VIEW nouvelles_ventes AS
SELECT id_vente, date, magasin, produit, quantite, prix_unitaire,
       chiffre_affaires, fichier_source
FROM (
    SELECT
        id_vente,
        CAST(date AS DATE)            AS date,
        magasin,
        produit,
        CAST(quantite AS INT)         AS quantite,
        CAST(prix_unitaire AS DOUBLE) AS prix_unitaire,
        CAST(quantite AS INT) * CAST(prix_unitaire AS DOUBLE) AS chiffre_affaires,
        fichier_source,
        ROW_NUMBER() OVER (PARTITION BY id_vente ORDER BY charge_le DESC) AS rn
    FROM bronze_ventes
    WHERE id_vente IS NOT NULL
      AND CAST(quantite AS INT) > 0
      AND CAST(prix_unitaire AS DOUBLE) > 0
)
WHERE rn = 1
""")

print("Lignes source :", spark.table("nouvelles_ventes").count())
print("Lignes dans silver :", spark.table("silver_ventes").count())

In [0]:
resultat = spark.sql("""
    MERGE INTO silver_ventes AS cible
    USING nouvelles_ventes AS source
    ON cible.id_vente = source.id_vente
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
display(resultat)

In [0]:
spark.sql("UPDATE silver_ventes SET quantite = 99 WHERE id_vente = '1-0'")
display(spark.sql("SELECT id_vente, quantite, prix_unitaire FROM silver_ventes WHERE id_vente = '1-0'"))

In [0]:
spark.sql("""
    MERGE INTO silver_ventes AS cible
    USING nouvelles_ventes AS source
    ON cible.id_vente = source.id_vente
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
display(spark.sql("SELECT id_vente, quantite, prix_unitaire FROM silver_ventes WHERE id_vente = '1-0'"))

In [0]:
display(spark.sql("DESCRIBE HISTORY silver_ventes"))

In [0]:
historique = spark.sql("DESCRIBE HISTORY silver_ventes")
display(historique.select("version", "timestamp", "operation").orderBy("version"))

# on retrouve automatiquement la version du dernier UPDATE
v = (historique.filter("operation = 'UPDATE'")
               .orderBy("version", ascending=False)
               .first()["version"])
print("Version de l'UPDATE :", v)

display(spark.sql(f"""
    SELECT id_vente, quantite
    FROM silver_ventes VERSION AS OF {v}
    WHERE id_vente = '1-0'
"""))